importing what is needed

In [25]:
import librosa
import os
import numpy as np
import collections

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

lets first load the data using librosa 

In [26]:
file_paths = librosa.util.find_files('/home/pranav/Downloads/archive/audio_speech_actors_01-24',recurse=True)

spectrograms_list = []
labels_list = []
for file_path in file_paths :
    #load the audio using librosa
    y , sr = librosa.load(file_path)

    #extract the file label from the name as the 3rd number tells the label 
    # use os lib for the task 
    file_name = os.path.basename(file_path)
    file_name = file_name.split('-')  
    label = int(file_name[2]) - 1 #pyotrch need indexing from 0
    

    #now we need to convert that audio to mel_spectrogram . no of bands is taken 64 as it is reasonable commonly used 
    mel = librosa.feature.melspectrogram(y=y,sr=sr,n_mels = 64)

    #we need to convert this mel from power values to decibles as power causes a great range of values 
    mel_db = librosa.power_to_db(mel,ref = np.max)  #np.max is passed as a reference point for the dB conversion.

    spectrograms_list.append(mel_db)
    labels_list.append(label)

print(spectrograms_list[0].shape)

print(len(spectrograms_list))

(64, 143)
1440


In [27]:
#lets check the dimensions of each spectrogram 
shapes = [s.shape for s in spectrograms_list]
print(set(shapes))

{(64, 130), (64, 136), (64, 194), (64, 133), (64, 200), (64, 197), (64, 206), (64, 212), (64, 151), (64, 209), (64, 154), (64, 160), (64, 157), (64, 163), (64, 169), (64, 166), (64, 172), (64, 184), (64, 187), (64, 193), (64, 190), (64, 196), (64, 141), (64, 199), (64, 138), (64, 144), (64, 150), (64, 205), (64, 202), (64, 147), (64, 153), (64, 156), (64, 220), (64, 159), (64, 174), (64, 177), (64, 180), (64, 186), (64, 131), (64, 183), (64, 189), (64, 128), (64, 134), (64, 192), (64, 137), (64, 143), (64, 140), (64, 146), (64, 207), (64, 210), (64, 149), (64, 213), (64, 216), (64, 161), (64, 167), (64, 164), (64, 170), (64, 176), (64, 228), (64, 173), (64, 179), (64, 182), (64, 127)}


In [28]:
#the time lengths  of all the spectrograms are not same . it should be same as CNN needs a fixed amount of input
time_lengths = [s.shape[1] for s in spectrograms_list]
print(max(time_lengths))
print(min(time_lengths))

228
127


to bring the time length same lets take the mod value in the beginning we can later change according to the best accuracy giver

In [29]:
counter = collections.Counter(time_lengths)
print(counter.most_common(5))
#161 is 81 times

[(161, 81), (154, 68), (153, 67), (156, 67), (151, 66)]


whatever is >161 will be sliced and <161 will be padded with 0

In [30]:
spectrograms_new=[]
for sp in spectrograms_list:
    if sp.shape[1] > 161 :
        spectrograms_new.append(sp[:,:161])
    if sp.shape[1] < 161:
        spectrograms_new.append(np.pad(sp, ((0,0), (0, 161-sp.shape[1]))))
    elif sp.shape[1] == 161:
        spectrograms_new.append(sp)
time_lengths_new = [s.shape[1] for s in spectrograms_new]
print(max(time_lengths_new))
print(min(time_lengths_new))
print(spectrograms_new[0].shape)

161
161
(64, 161)


now the data needs to be channelized before applying CNN

In [31]:
X = np.array(spectrograms_new)  # shape: (1440, 64, 161)
X = X[:, np.newaxis, :, :]      # shape: (1440, 1, 64, 161) this has 1 new dimension that denotes the no. of channels
y = np.array(labels_list)

next 3 cells are almost same as the recommendation system model

In [32]:
class InteractionDataset (Dataset) :
    def __init__(self,X,y):
        super().__init__()
        self.X = torch.tensor(X , dtype= torch.float32)
        self.y = torch.tensor(y , dtype= torch.long)

    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, index):
        return self.X[index] , self.y[index]


In [33]:
from sklearn.model_selection import train_test_split

X_train, X_test = train_test_split(X,  test_size=0.2, random_state=42)


y_train, y_test = train_test_split(y, test_size=0.2, random_state=42)

In [34]:
train_dataset = InteractionDataset(X_train, y_train)
test_dataset = InteractionDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

writing a AudioCNN class that contains the layers of CNN and neural linear layers along with the forward

In [35]:
class AudioCNN(nn.Module):
    def __init__(self):
        super().__init__()
        #each layer doubles the number of filters.so we take in doubling from 16 to 32 to 64 any other can also be taken will check 
        #what happens on changing it later
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3)
        self.conv2 = nn.Conv2d(16, 32 , kernel_size=3)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3)
        #i have used max pool first will try with avg pool later 
        self.pool = nn.MaxPool2d(2)
        self.flatten = nn.Flatten()

        self.L1 = nn.Linear(6912, 128)  #checked by putting random value before and getting the value from the error
        self.L2 = nn.Linear(128, 8)  
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = self.flatten(x)
        x = self.relu(self.L1(x))
        #print(x.shape)
        #we are not putting softmax as the CEL function applies it by itself
        return self.L2(x)

the below is a fake batch just to get the no of input nodes for the first lineat layer

In [36]:
model = AudioCNN()
sample = torch.randn(32, 1, 64, 161)  # fake batch
model(sample)

tensor([[ 0.0230, -0.0486, -0.0144,  0.0152, -0.0654,  0.1324,  0.0893,  0.0342],
        [ 0.0356, -0.0330, -0.0148,  0.0194, -0.0647,  0.1438,  0.0714,  0.0294],
        [ 0.0283, -0.0162,  0.0097,  0.0288, -0.0420,  0.1368,  0.0639,  0.0235],
        [ 0.0322, -0.0189,  0.0010,  0.0240, -0.0708,  0.1353,  0.0681,  0.0288],
        [ 0.0336, -0.0120,  0.0190,  0.0414, -0.0585,  0.1267,  0.0724,  0.0462],
        [ 0.0354, -0.0281, -0.0037,  0.0359, -0.0629,  0.1342,  0.0785,  0.0228],
        [ 0.0510, -0.0314,  0.0038,  0.0193, -0.0571,  0.1413,  0.0641,  0.0372],
        [ 0.0215, -0.0223, -0.0233,  0.0251, -0.0519,  0.1323,  0.0778,  0.0478],
        [ 0.0384, -0.0134, -0.0005,  0.0223, -0.0629,  0.1453,  0.0703,  0.0348],
        [ 0.0459, -0.0151, -0.0074,  0.0322, -0.0619,  0.1346,  0.0779,  0.0240],
        [ 0.0456, -0.0413, -0.0221,  0.0346, -0.0668,  0.1234,  0.0680,  0.0344],
        [ 0.0297, -0.0075, -0.0061,  0.0264, -0.0836,  0.1340,  0.0733,  0.0349],
        [ 0.0490

now same as recommender system model we need a train function 

In [37]:
def train (model , dataloader , epochs ):
    #lets move the model to device gpu 
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    #defining the optimiser to be used
    optimiser = torch.optim.Adam(model.parameters(),lr = 0.001)

    #defining the loss function CEL as there are multiple categories
    loss_fun = nn.CrossEntropyLoss()

    #creating  a list of the losses per epoch
    losses_epochs = []
    for epoch in range(epochs) :
        
        total_loss = 0
        
        for  X,labels in dataloader:

            #now we run the following functions on a batch of the epoch

            #moving the batch parameters to the cuda
            X = X.to(device)
            labels = labels.to(device)

            # we need to zero all the accumulated gradients in previous iterations
            optimiser.zero_grad()

            #now lets get the prediction
            pred = model(X)     

            #compute the loss 
            loss = loss_fun(pred,labels)

            #compute gradient
            loss.backward()

            #optimiser 
            optimiser.step()

            total_loss += loss.item()     #loss.item() gives the loss of one batch 

            #appending the loss of a batch to the list of losses so that we may later plot a curve analyse the descent
        losses_epochs.append(total_loss)
        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")
    return losses_epochs
                    



now we create an evaluater function that will check the accuracy by correct/total

In [38]:
def evaluate(model, dataloader):
    model.eval()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    correct = 0
    total = 0
    
    with torch.no_grad():
        for X, labels in dataloader:
            X = X.to(device)
            labels = labels.to(device)
            pred = model(X)
            predictions = torch.argmax(pred,dim = 1)
            
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    
    return correct / total

In [39]:
cnn_model = AudioCNN()
losses = train(cnn_model, train_loader, epochs=10)
accuracy = evaluate(cnn_model, test_loader)
print(f"CNN Accuracy: {accuracy:.4f}")

Epoch 1, Loss: 82.5782
Epoch 2, Loss: 70.0030
Epoch 3, Loss: 66.6304
Epoch 4, Loss: 60.9006
Epoch 5, Loss: 53.6525
Epoch 6, Loss: 50.4633
Epoch 7, Loss: 46.0320
Epoch 8, Loss: 41.9769
Epoch 9, Loss: 38.1989
Epoch 10, Loss: 35.1910
CNN Accuracy: 0.5382
